# BERT 全量微调对照实验（ALL_Funning）

本Notebook使用 `Bert_Config.py` 的统一配置与通用函数，进行全量微调（非LoRA）训练与评估。

In [1]:
"""
第一部分：导入必要的库
"""

import os
import time

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from transformers import BertConfig, BertForSequenceClassification, get_linear_schedule_with_warmup
from torch.optim import AdamW
from tqdm import tqdm

from Bert_Config import CONFIG, setup_seed, build_tokenizer, generate_data, get_save_path

print("✅ 所有库导入完成")

C:\Users\19836\miniconda3\envs\py8\lib\site-packages\tqdm\auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ 所有库导入完成


In [2]:
"""
第二部分：统一配置
"""

MODEL_PATH = CONFIG["BERT_MODEL_PATH"]
DATA_PATH = CONFIG["DATA_DIR"]
EPOCHS = CONFIG["EPOCHS"]
LEARNING_RATE = 1e-5
BATCH_SIZE = CONFIG["BATCH_SIZE"]
MAX_LENGTH = CONFIG["MAX_LENGTH"]
DROPOUT = CONFIG["DROPOUT"]
NUM_CLASSES = CONFIG["NUM_CLASSES"]
WEIGHT_DECAY = CONFIG["WEIGHT_DECAY"]
FREEZE_BERT = False
RANDOM_SEED = CONFIG["RANDOM_SEED"]
SAVE_PATH = get_save_path("all")


if not os.path.exists(MODEL_PATH):
    print(f"⚠️  模型文件夹不存在: {MODEL_PATH}")
else:
    print(f"✅ 模型路径检查通过: {MODEL_PATH}")

EXP_NAME = "全参数微调"
print("=" * 50)
print(f"实验配置: {EXP_NAME}")
print("=" * 50)
print(f"BERT模型路径: {MODEL_PATH}")
print(f"数据目录: {DATA_PATH}")
print(f"训练轮数: {EPOCHS}")
print(f"学习率: {LEARNING_RATE}")
print(f"批次大小: {BATCH_SIZE}")
print(f"最大文本长度: {MAX_LENGTH}")
print(f"dropout率 (DROPOUT): {DROPOUT}")
print(f"分类类别数: {NUM_CLASSES}")
print(f"权重衰减: {WEIGHT_DECAY}")
print(f"是否冻结BERT (FREEZE_BERT): {FREEZE_BERT}")
print(f"随机种子: {RANDOM_SEED}")
print(f"模型保存路径: {SAVE_PATH}")
print("=" * 50)

print("✅ 配置加载完成")

✅ 模型路径检查通过: ../bert-base-chinese
实验配置: 全参数微调
BERT模型路径: ../bert-base-chinese
数据目录: ../waimai.csv
训练轮数: 5
学习率: 3e-05
批次大小: 32
最大文本长度: 256
dropout率 (DROPOUT): 0.1
分类类别数: 2
权重衰减: 0.01
是否冻结BERT (FREEZE_BERT): False
随机种子: 42
模型保存路径: ./bert_all_checkpoint
✅ 配置加载完成


In [3]:
"""
第三部分：设置随机种子
"""

setup_seed(RANDOM_SEED)
print("✅ 随机种子设置完成")

✅ 随机种子设置完成


In [4]:
"""
第四部分：初始化分词器
"""

tokenizer = build_tokenizer(MODEL_PATH)
print("✅ 分词器加载完成")
print(f"   - 词汇表大小: {len(tokenizer.vocab)}")

✅ 分词器加载完成
   - 词汇表大小: 21128


In [5]:
"""
第五部分：准备训练数据
"""

train_dataset = generate_data("train", DATA_PATH, tokenizer, MAX_LENGTH, RANDOM_SEED)
val_dataset = generate_data("val", DATA_PATH, tokenizer, MAX_LENGTH, RANDOM_SEED)

train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=BATCH_SIZE)

print("✅ 训练数据准备完成")
print(f"   - 训练批次数: {len(train_dataloader)}")
print(f"   - 验证批次数: {len(val_dataloader)}")

数据基本信息
数据总量: 11987

标签分布:
0    7987
1    4000
Name: label, dtype: int64

数据示例（前3条）:
   label        review
0      1  很快，好吃，味道足，量大
1      1  没有送水没有送水没有送水
2      1      非常快，态度好。

数据分割结果
训练集大小: 9589 (80.0%)
验证集大小: 1199 (10.0%)
测试集大小: 1199 (10.0%)

✅ 数据加载和预处理完成
✅ 训练数据准备完成
   - 训练批次数: 300
   - 验证批次数: 38


In [6]:
"""
第六部分：定义并初始化模型
"""

config = BertConfig.from_pretrained(MODEL_PATH)
config.hidden_dropout_prob = DROPOUT
config.attention_probs_dropout_prob = DROPOUT
config.num_labels=NUM_CLASSES

model = BertForSequenceClassification.from_pretrained(
    MODEL_PATH,
    config=config,
)

if FREEZE_BERT:
    for param in model.bert.parameters():
        param.requires_grad = False
    print("bert主干参数已经被冻结")

print("✅ BERT 模型初始化完成")
print(f"   - 模型参数数量: {sum(p.numel() for p in model.parameters()):,}")
print(f"   - 可训练参数数量: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

Some weights of the model checkpoint at ../bert-base-chinese were not used when initializing BertForSequenceClassification: ['cls.predictions.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.decoder.weight', 'cls.predictions.transform.dense.weight', 'cls.seq_relationship.weight', 'cls.seq_relationship.bias', 'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.transform.dense.bias']
- This IS expected if you are initializing BertForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of BertForSequenceClassification were not initialized from the model checkpoint

✅ BERT 模型初始化完成
   - 模型参数数量: 102,269,186
   - 可训练参数数量: 102,269,186


In [7]:
"""
第七部分：配置训练环境
"""

use_cuda = torch.cuda.is_available()
device = torch.device("cuda" if use_cuda else "cpu")
print(f"使用设备: {device}")

criterion = nn.CrossEntropyLoss()
optimizer = AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    eps=1e-8,
)

# 添加学习率调度器（线性调度+预热）
total_steps = len(train_dataloader) * EPOCHS
warmup_steps = int(total_steps * 0.1)  # 10%预热
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

if use_cuda:
    model = model.cuda()
    criterion = criterion.cuda()

print("✅ 训练环境配置完成")
print(f"   - 总训练步数: {total_steps}")
print(f"   - 预热步数: {warmup_steps}")

使用设备: cpu
✅ 训练环境配置完成


In [8]:
"""
第八部分：评估函数和早停机制
"""

def evaluate_bert(model, dataloader, criterion, device):
    """
    评估BERT模型在数据集上的性能
    适配BERT的输入格式（input_ids, attention_mask）
    """
    model.eval()
    total_loss = 0.0
    all_labels = []
    all_preds = []
    
    with torch.no_grad():
        for inputs, labels in dataloader:
            labels = labels.to(device)
            mask = inputs["attention_mask"].to(device)
            input_ids = inputs["input_ids"].squeeze(1).to(device)
            
            outputs = model(input_ids=input_ids, attention_mask=mask)
            logits = outputs.logits
            loss = criterion(logits, labels)
            total_loss += loss.item() * labels.size(0)
            
            preds = logits.argmax(dim=1)
            all_labels.extend(labels.cpu().numpy().tolist())
            all_preds.extend(preds.cpu().numpy().tolist())
    
    avg_loss = total_loss / len(dataloader.dataset)
    acc = accuracy_score(all_labels, all_preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        all_labels, all_preds, average="binary", zero_division=0
    )
    return acc, precision, recall, f1, avg_loss


class EarlyStopping:
    def __init__(self, patience=3, min_delta=0.001):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_score = None
        
    def __call__(self, val_score):
        if self.best_score is None:
            self.best_score = val_score
        elif val_score < self.best_score + self.min_delta:
            self.counter += 1
            if self.counter >= self.patience:
                return True
        else:
            self.best_score = val_score
            self.counter = 0
        return False


def save_model(model, save_name):
    if not os.path.exists(SAVE_PATH):
        os.makedirs(SAVE_PATH)
        print(f"✅ 创建模型保存目录: {SAVE_PATH}")

    save_file = os.path.join(SAVE_PATH, save_name)
    torch.save(model.state_dict(), save_file)
    print(f"✅ 模型已保存: {save_file}")

print("✅ 评估函数和早停机制定义完成")

✅ 模型保存函数定义完成


In [ ]:
"""
第九部分：训练模型
"""

print("\n开始训练...")
train_start_time = time.time()
best_dev_acc = 0
early_stopping = EarlyStopping(patience=3, min_delta=0.001)

for epoch_num in range(EPOCHS):
    model.train()
    total_acc_train = 0
    total_loss_train = 0

    for train_input, train_label in tqdm(
        train_dataloader,
        desc=f"Epoch {epoch_num + 1}/{EPOCHS} [训练]",
    ):
        train_label = train_label.to(device)
        mask = train_input["attention_mask"].to(device)
        input_id = train_input["input_ids"].squeeze(1).to(device)

        outputs = model(
            input_ids=input_id,
            attention_mask=mask,
        )
        logits = outputs.logits
        batch_loss = criterion(logits, train_label)
        # 修复损失计算：batch_loss是平均损失，需要乘以批次大小得到总损失
        total_loss_train += batch_loss.item() * train_label.size(0)

        acc = (logits.argmax(dim=1) == train_label).sum().item()
        total_acc_train += acc

        model.zero_grad()
        batch_loss.backward()
        # 添加梯度裁剪，防止梯度爆炸
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        # 更新学习率
        scheduler.step()

    # 使用评估函数进行验证，获得完整的评估指标
    val_acc, val_precision, val_recall, val_f1, val_loss = evaluate_bert(
        model, val_dataloader, criterion, device
    )
    
    # 计算训练集的平均损失和准确率
    train_loss_avg = total_loss_train / len(train_dataset)
    train_acc_avg = total_acc_train / len(train_dataset)
    
    print(
        f"[Epoch {epoch_num + 1}/{EPOCHS}] "
        f"Train Loss: {train_loss_avg:.4f} | "
        f"Train Acc: {train_acc_avg:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_acc:.4f} | "
        f"Val Precision: {val_precision:.4f} | "
        f"Val Recall: {val_recall:.4f} | "
        f"Val F1: {val_f1:.4f}"
    )

    if val_acc > best_dev_acc:
        best_dev_acc = val_acc
        save_model(model, "best.pt")
        print(f"   🎯 发现更好的模型！验证准确率: {best_dev_acc:.3f}")

    # 检查早停
    if early_stopping(val_acc):
        print(f"   ⏹️  早停触发，验证准确率连续{early_stopping.patience}个epoch未提升")
        break

train_end_time = time.time()
training_time_sec = train_end_time - train_start_time
training_time_min = training_time_sec / 60

save_model(model, "last.pt")

print("\n✅ 训练完成！")
print(f"   - 最佳验证准确率: {best_dev_acc:.3f}")
print(f"   - 最佳模型已保存: {os.path.join(SAVE_PATH, 'best.pt')}")
print(f"   - 最后模型已保存: {os.path.join(SAVE_PATH, 'last.pt')}")


开始训练...


Epoch 1/5 [训练]:   0%|          | 0/300 [00:11<?, ?it/s]


In [ ]:
"""
第十部分：测试集评估
"""

test_dataset = generate_data("test", DATA_PATH, tokenizer, MAX_LENGTH, RANDOM_SEED)
test_dataloader = DataLoader(test_dataset, batch_size=BATCH_SIZE)

# 加载最佳模型
print("\n加载最佳模型进行测试集评估...")
model.load_state_dict(torch.load(os.path.join(SAVE_PATH, "best.pt")))

# 使用评估函数进行测试集评估，确保评估逻辑一致
test_acc, test_precision, test_recall, test_f1, test_loss = evaluate_bert(
    model, test_dataloader, criterion, device
)

print("\n测试集评估结果:")
print(f"  - Loss: {test_loss:.3f}")
print(f"  - Accuracy: {test_acc:.3f}")
print(f"  - Precision: {test_precision:.3f}")
print(f"  - Recall: {test_recall:.3f}")
print(f"  - F1 Score: {test_f1:.3f}")
print(f"   - 训练时间: {training_time_sec:.1f} 秒 ({training_time_min:.2f} 分钟)")

print(f"\n🎉 最终测试准确率: {test_acc:.3f}")